Imports / Configurações

In [1]:
import pandas as pd
import numpy as np
import sys
import pickle

sys.path.append('..')
from src.text_processing import CustomTFIDF
from src.numpy_models import NumpyDNN

In [ ]:
import math

def encode_labels(labels):
    unique_labels = sorted(set(labels))
    label_to_idx = {label: i for i, label in enumerate(unique_labels)}
    idx_to_label = {i: label for i, label in enumerate(unique_labels)}
    Y = np.zeros((len(labels), len(unique_labels)))
    for i, label in enumerate(labels):
        Y[i, label_to_idx[label]] = 1
    print(f"Classes detetadas: {unique_labels}")
    return Y, label_to_idx, idx_to_label

## TF-IDF

In [2]:
df = pd.read_csv("../data/dataset_completo.csv")

corte = int(len(df) * 0.8)      # split 80/20

df_treino = df.iloc[:corte]
df_val = df.iloc[corte:]

textos_treino = df_treino['text'].values
labels_treino = df_treino['label'].values

tfidf = CustomTFIDF(max_features=1000) 

tfidf.fit(textos_treino)
X_train = tfidf.transform(textos_treino)
Y_train, class_map, reverse_class_map = encode_labels(labels_treino)

print(f"Formato da Matriz de Entrada (X_train): {X_train.shape} -> (Exemplos, Palavras)")
print(f"Formato da Matriz de Saída (Y_train): {Y_train.shape} -> (Exemplos, Classes)")

Vocabulário criado com 1000 palavras
Classes detetadas: ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
Formato da Matriz de Entrada (X_train): (481, 1000) -> (Exemplos, Palavras)
Formato da Matriz de Saída (Y_train): (481, 5) -> (Exemplos, Classes)


# Modelo NumPy

In [3]:
input_size = X_train.shape[1]       # O tamanho do vocabulário TF-IDF
hidden_size = 64                    # Outros valores: 32, 128
output_size = Y_train.shape[1]      # 5 classes

modelo_dnn = NumpyDNN(input_size=input_size, 
                      hidden_size=hidden_size, 
                      output_size=output_size, 
                      learning_rate=0.5) # Se a loss não descer, aumentar LR para 0.5 ou 1.0 (base = 0.1)

modelo_dnn.train(X_train, Y_train, epochs=3000)     # mudar epochs consoante necessário

A iniciar o treino da rede neuronal
Época 0 | Loss: 1.6094 | Accuracy: 20.17%
Época 100 | Loss: 1.6061 | Accuracy: 21.83%
Época 200 | Loss: 1.5926 | Accuracy: 21.83%
Época 300 | Loss: 1.4398 | Accuracy: 40.12%
Época 400 | Loss: 1.2147 | Accuracy: 40.12%
Época 500 | Loss: 1.1584 | Accuracy: 41.79%
Época 600 | Loss: 1.1292 | Accuracy: 56.96%
Época 700 | Loss: 1.0884 | Accuracy: 73.39%
Época 800 | Loss: 1.0103 | Accuracy: 78.59%
Época 900 | Loss: 1.0123 | Accuracy: 40.96%
Época 1000 | Loss: 0.8841 | Accuracy: 59.25%
Época 1100 | Loss: 0.7650 | Accuracy: 70.48%
Época 1200 | Loss: 0.6015 | Accuracy: 77.96%
Época 1300 | Loss: 0.6999 | Accuracy: 69.65%
Época 1400 | Loss: 0.4557 | Accuracy: 87.32%
Época 1500 | Loss: 0.3099 | Accuracy: 93.14%
Época 1600 | Loss: 0.2507 | Accuracy: 96.67%
Época 1700 | Loss: 0.1850 | Accuracy: 97.71%
Época 1800 | Loss: 0.1586 | Accuracy: 97.92%
Época 1900 | Loss: 0.1081 | Accuracy: 99.79%
Época 2000 | Loss: 0.0891 | Accuracy: 100.00%
Época 2100 | Loss: 0.0746 | Ac

In [4]:
# Validação

X_val = tfidf.transform(df_val['text'].values)
previsoes_val = modelo_dnn.predict(X_val)
reais_val = np.argmax(encode_labels(df_val['label'].values)[0], axis=1)

accuracy_val = np.mean(previsoes_val == reais_val) * 100
print(f"Accuracy real nos dados de Validação: {accuracy_val:.2f}%")

Classes detetadas: ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
Accuracy real nos dados de Validação: 96.69%


Guardar modelo

In [9]:
with open('../modelos/modelo_A.pkl', 'wb') as f:
    pickle.dump({
        'tfidf': tfidf,
        'modelo': modelo_dnn,
        'reverse_map': reverse_class_map
    }, f)
print("Modelo guardado com sucesso!")

Modelo guardado com sucesso!
